# CQRS Q6 v7 — admission trước transaction và smoke O/A/B

Upload notebook → Kaggle CPU / Internet On → Run All. Source được nhúng sẵn,
không cần commit/push hoặc upload repo riêng. **Mặc định chỉ smoke O/A/B cho
query-heavy và PCN; chưa chạy benchmark dài.** Bản v6 trước đó FAIL trên native.
Bản v7 giữ cache aggregate theo transactional epoch/ACL, đưa hàng chờ ra trước
transaction để waiter không giữ connection/snapshot. Thời gian chờ trừ khỏi
deadline projected1250ms/RYW500ms; SQL200ms/maxWait50ms và pool/rate giữ nguyên.
Freshness vẫn kiểm broker và full registry/read-row mỗi2s, dùng0 parallel worker
để hạn chế tranh CPU với query; không đổi lag/age limits. Thêm diagnostics
admission/acquire/setup và sampled cache hit/miss, không chứa SQL/actor/params.
Các migration additive hiện có chỉ apply vào trial UUID do notebook tạo;
v7 không thêm migration mới. Kiểm local không thay native v7. Giữ mọi FAIL,
chưa chạy full trước khi đối soát smoke mới. Không chạy lại nguyên notebook v1–v6.

O là Git blob tại `3e88d7e57fbf4f74da363d320f286244237a4389`. A/B dùng cùng
snapshot source CQRS chưa commit, khác selector authoritative/projection;
producer/projector bật cả A/B, shadow tắt. Mỗi lượt tạo trial UUID riêng,
đúng ba DB; không dùng DB ngoài. B phải qua HTTP preflight đúng generation.
Query-heavy còn kiểm nguồn đọc từng request; unbilled luôn authoritative.

A/B bật diagnostics, O giữ source gốc. ZIP có `read-diagnostics.jsonl` đã lọc
stage/reason/Prisma code/SQLSTATE và ID nối với raw query-heavy request.
Tối đa512 event/process rồi1 marker giới hạn; lỗi collector/thiếu log được ghi
trong report. Không xuất log riêng, token, query params hoặc thông báo lỗi thô.
`elapsedMs`/`stageMs` là thời gian chẩn đoán có overhead, không phải số đo NFR.
Sau khi B dừng tải, thu thêm `query-plans.json` cho counts/calendar bằng chính
SQL của projected reader, trong transaction READ ONLY có budget riêng.
Collector đo SQL khi không dùng cache để chẩn đoán miss, không phải thời gian hit.
Plan đã lọc bỏ filter/params/SQL text; chỉ giữ loại node và số liệu.
Đây là plan sau tải, không phải phép đo dưới cùng concurrency; lỗi thu plan
được ghi UNAVAILABLE/PARTIAL và không đổi kết quả smoke.

Giữ `RUN_FULL=False` để kiểm smoke trước (dự kiến15–30 phút kể cả setup).
Sau khi xem ZIP smoke, chọn `True` trong một phiên mới để đo dài: mỗi workload
smoke O/A/B → O×3 → AB,BA,AB. PCN100 users/1800s, query-heavy100k rows,
rate12/s/warmup120s/steady800s. Cả hai workload đầy đủ khoảng7–9 giờ kể cả setup;
thời gian thực tùy máy. Không retry, giảm tải, tăng quota hoặc bỏ mẫu lỗi.

**Không nghiệm thu hiệu năng/capacity từ notebook này.** Host Kaggle dùng chung,
chưa cưỡng chế ngân sách tổng bằng nhau; report giữ `INCONCLUSIVE`, default
authoritative. Có raw HTTP/errors/CPU/DB/lag/fallback để đánh giá sau; chưa có
bộ chốt gate Q6 tự động. Không tự bật projection ở môi trường đang dùng.

In [ ]:
RUN_FULL = False
WORKLOADS = ['query-heavy', 'pcn']
import base64, hashlib, importlib.util, io, json, os, platform, subprocess, sys, urllib.request, uuid
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import display, FileLink
assert platform.system() == 'Linux' and Path('/kaggle/working').is_dir(), 'Chạy trong Kaggle CPU notebook'
assert WORKLOADS and len(set(WORKLOADS)) == len(WORKLOADS) and set(WORKLOADS) <= {'query-heavy','pcn'}
OUTPUT = Path('/kaggle/working') / ('cqrs-q6-' + str(uuid.uuid4()))
OUTPUT.mkdir()
RESULTS = []

SOURCE_SHA256 = 'd9071b6629b656c5519e1fdb33bfe8fbffce03e9dc235d34f56311f373390fcf'
SOURCE_BASE64 = '...'
source_bytes = base64.b64decode(SOURCE_BASE64, validate=True)
assert hashlib.sha256(source_bytes).hexdigest() == SOURCE_SHA256
BOOT = Path('/tmp') / ('pet-cqrs-q6-bootstrap-' + str(uuid.uuid4()))
BOOT.mkdir(mode=0o700)
SOURCE_ZIP = BOOT / 'source.zip'
SOURCE_ZIP.write_bytes(source_bytes)
# Only bootstrap the verifier after the entire archive hash matches. It then
# checks every member/path/hash before extracting; never use extractall.
with ZipFile(io.BytesIO(source_bytes)) as archive:
    verifier = archive.read('harness/benchmarks/phase2/kaggle/cqrs_source.py')
verifier_path = BOOT / 'benchmarks/phase2/kaggle/cqrs_source.py'
verifier_path.parent.mkdir(parents=True)
verifier_path.write_bytes(verifier)
spec = importlib.util.spec_from_file_location('q6_bootstrap_verifier', verifier_path)
verifier_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(verifier_module)
verifier_module.extract_bundle(SOURCE_ZIP, BOOT/'verified', SOURCE_SHA256)
sys.path.insert(0, str(BOOT/'verified/harness/benchmarks/phase2/kaggle'))
from cqrs_benchmark import run
print({'source_archive_sha256':SOURCE_SHA256,'source_verified':True,'run_full':RUN_FULL,'capacity_accepted':False})

## Cài công cụ thử
Tái sử dụng setup Node24.16.0/Locust2.44.4 và stack native của Q0. Log/DB/credentials nằm trong `/tmp`; không đưa vào ZIP kết quả.

In [ ]:
assert os.geteuid() == 0, 'Cell cài apt cần quyền root của phiên Kaggle'
SETUP = Path('/tmp') / ('pet-phase2-setup-' + str(uuid.uuid4()))
SETUP.mkdir(mode=0o700)
def setup_command(args, **kwargs):
    with (SETUP / 'setup.log').open('a') as log:
        result = subprocess.run(args, stdout=log, stderr=subprocess.STDOUT, **kwargs)
    assert result.returncode == 0, 'Setup thất bại; xem setup.log riêng trong /tmp, không công bố log trước khi kiểm secret'

# Chặn auto-start service mặc định của apt; chỉ restore file do cell này tạo.
policy = Path('/usr/sbin/policy-rc.d')
created_policy = not policy.exists()
if created_policy:
    policy.write_text('#!/bin/sh\nexit 101\n'); policy.chmod(0o755)
try:
    setup_command(['apt-get', 'update', '-qq'])
    setup_command(['apt-get', 'install', '-y', '-qq', '--no-install-recommends',
                   'postgresql', 'postgresql-client', 'libpq-dev', 'rabbitmq-server',
                   'redis-server', 'nginx', 'ca-certificates', 'xz-utils'])
finally:
    if created_policy and policy.read_text() == '#!/bin/sh\nexit 101\n': policy.unlink()

assert platform.machine() in ('x86_64','aarch64'), 'Kiến trúc Node chưa hỗ trợ'
arch = 'x64' if platform.machine() == 'x86_64' else 'arm64'
name = f'node-v24.16.0-linux-{arch}.tar.xz'
origin = 'https://nodejs.org/dist/v24.16.0/'
sums = urllib.request.urlopen(origin+'SHASUMS256.txt', timeout=30).read().decode()
expected = next(line.split()[0] for line in sums.splitlines() if line.split()[-1] == name)
node_archive = SETUP / name
urllib.request.urlretrieve(origin+name, node_archive)
assert hashlib.sha256(node_archive.read_bytes()).hexdigest() == expected
setup_command(['tar','-xJf',str(node_archive),'-C',str(SETUP)])  # official archive verified above
os.environ['PATH'] = str(SETUP / name.removesuffix('.tar.xz') / 'bin') + ':' + os.environ['PATH']
setup_command([sys.executable,'-m','pip','install','-q','--no-cache-dir','locust==2.44.4','psutil==7.0.0'])
VERSIONS = {}
for program, args in {'node':['--version'], 'psql':['--version'], 'redis-server':['--version'], 'nginx':['-v']}.items():
    VERSIONS[program] = subprocess.check_output([program,*args], stderr=subprocess.STDOUT, text=True).strip()
VERSIONS['packages'] = subprocess.check_output(['dpkg-query','-W','postgresql','rabbitmq-server','redis-server','nginx'], text=True).splitlines()
print({'versions':VERSIONS,'node_archive_sha256':expected})
(OUTPUT/'setup-versions.json').write_text(json.dumps({'versions':VERSIONS,'node_archive_sha256':expected},indent=2))

## Chạy tuần tự
Dừng ngay khi workload lỗi; giữ kết quả lỗi. Không suy smoke thành nghiệm thu PCN hoặc Q6.

In [ ]:
for workload in WORKLOADS:
    result = run(SOURCE_ZIP, SOURCE_SHA256, OUTPUT/workload, workload=workload, full=RUN_FULL)
    RESULTS.append({'workload':workload, **result})
    print({'workload':workload,'status':result['status'],'decision':result.get('decision','INCONCLUSIVE')})
    if result['status'] not in ('SMOKE_ONLY','MEASUREMENTS_RECORDED'):
        break
(OUTPUT/'summary.json').write_text(json.dumps({'results':RESULTS,'run_full':RUN_FULL,'capacity_accepted':False,'decision':'INCONCLUSIVE'},indent=2))
for result in RESULTS:
    for trial in result.get('trials',[]):
        print({k:trial.get(k) for k in ('workload','name','configuration','phase','status')})

## Tải ZIP kết quả
Gửi ZIP để đối soát raw/hash/các gate. Chỉ export allowlist; không chứa source, context, token, fixture, log riêng hoặc DB. Cả FAIL/STOPPED đều được giữ.

In [ ]:
allowed = ['summary.json','setup-versions.json']
trial_names = ['smoke-O','smoke-A','smoke-B','O-1','O-2','O-3','pair-1-A','pair-1-B','pair-2-B','pair-2-A','pair-3-A','pair-3-B']
for workload in WORKLOADS:
    allowed += [workload+'/'+name for name in ('environment.json','run.json','trials/workflow.json')]
    for trial in trial_names:
        allowed += [workload+'/trials/'+trial+'/'+name for name in (
            'report.json','requests.jsonl','resources.jsonl','database.jsonl','cqrs.jsonl','read-diagnostics.jsonl','query-plans.json',
            'manifest.json','pcn-report.json','http.csv','http-summary.json',
            'load-final.json','generator.json','reconciliation.json')]
files = []
for name in allowed:
    path = OUTPUT/name
    if path.is_file() and not path.is_symlink() and path.resolve().is_relative_to(OUTPUT.resolve()):
        files.append({'path':name,'bytes':path.stat().st_size,'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
(OUTPUT/'result-manifest.json').write_text(json.dumps({'source_archive_sha256':SOURCE_SHA256,'files':files,'decision':'INCONCLUSIVE','capacity_accepted':False},indent=2))
archive_out = OUTPUT.with_suffix('.zip')
with ZipFile(archive_out,'x',ZIP_DEFLATED) as archive:
    for item in files:
        archive.write(OUTPUT/item['path'],item['path'])
    archive.write(OUTPUT/'result-manifest.json','result-manifest.json')
display(FileLink(str(archive_out)))
print({'download':str(archive_out),'capacity_accepted':False})